# Build a Database Advisor Agent with the DeepWiki Connector

You need a database for write-heavy local analytics. SQLite, DuckDB, and LevelDB are all strong contenders, but which one actually fits? Rather than reading documentation by hand, this notebook lets Mistral read their actual source code via the [DeepWiki](https://deepwiki.com) Connector and decide.

This notebook demonstrates the full [Mistral Connector](https://docs.mistral.ai/studio-api/connectors) lifecycle:

| Step | Operation | What happens |
|---|---|---|
| 1 | **Create** | Register a connector for each database candidate |
| 2 | **List** | Verify all three are registered |
| 3 | **Use** | Build an agent that compares them via their GitHub repos |
| 4 | **Update** | Mark the winner's connector as selected |
| 5 | **Delete** | Clean up the losing connectors |

> **API status:** This notebook uses `client.beta.connectors` and `client.beta.agents`. These are **beta** endpoints and may change.

Run cells top-to-bottom. A TypeScript version of the same agent is also available [here](./01-build-a-database-advisor-agent-typescript.md).

## Prerequisites

To complete this notebook, you will need:
- Python 3.9 or later
- A Mistral account and API key

## Environment setup

Install the Mistral Python SDK by running the cell below.

To complete this cookbook, you'll need a Mistral API key. In [Studio](https://console.mistral.ai), navigate to the [API keys section](https://console.mistral.ai/home?profile_dialog=api-keys), choose **Private and shared connectors** for **Connector access scope** and create a new API key. 

Set it before running the client cell using one of these options:

**Option 1 — environment variable** (recommended for local use):

```
MISTRAL_API_KEY=your-mistral-api-key
```

**Option 2 — enter it when prompted**: if `MISTRAL_API_KEY` is not already set in your environment, the next code cell will display a secure input field where you can paste your key directly.

In [12]:
%pip install mistralai --quiet

Note: you may need to restart the kernel to use updated packages.


In [13]:
import getpass
import os
import re

from mistralai.client import Mistral

if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Mistral API key: ")

client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

## Step 1 — Create one connector per candidate

Each connector points at the [DeepWiki](https://deepwiki.com) MCP server, which lets Mistral read and reason about any public GitHub repository. We create three named Connectors (one per candidate) so each one acts as a named slot the agent can query independently.

All three point at the same MCP server URL; the connector names are what distinguish them when the agent decides which tools to call.

In [14]:
DEEPWIKI_URL = "https://mcp.deepwiki.com/mcp"

candidates = [
    {"name": "showdown_sqlite",  "description": "DeepWiki connector — sqlite/sqlite"},
    {"name": "showdown_duckdb",  "description": "DeepWiki connector — duckdb/duckdb"},
    {"name": "showdown_leveldb", "description": "DeepWiki connector — google/leveldb"},
]

connectors = {}
for c in candidates:
    connector = await client.beta.connectors.create_async(
        name=c["name"],
        description=c["description"],
        server=DEEPWIKI_URL,
        visibility="private",
    )
    connectors[c["name"]] = connector
    print(f"Created: {connector.name}  (id={connector.id})")

Created: showdown_sqlite  (id=019f8eb1-a4f7-741f-89d8-ead5578f579a)
Created: showdown_duckdb  (id=019f8eb1-abf9-75c1-80f6-346460a1dc34)
Created: showdown_leveldb  (id=019f8eb1-b2ff-7571-9914-a073d82908d4)


## Step 2 — List to verify

Confirm all three connectors are registered before proceeding.

View your registered Connectors in [Studio](https://console.mistral.ai/build/connectors).

In [15]:
page = await client.beta.connectors.list_async(page_size=50)
showdown = [c for c in page.items if c.name.startswith("showdown_")]

print(f"{len(showdown)} showdown connectors registered:\n")
for c in showdown:
    print(f"  {c.name:<22}  {c.description}")

3 showdown connectors registered:

  showdown_sqlite         DeepWiki connector — sqlite/sqlite
  showdown_duckdb         DeepWiki connector — duckdb/duckdb
  showdown_leveldb        DeepWiki connector — google/leveldb


## Step 3 — Build the comparison agent

We create a Mistral agent with all three connectors attached. Its instructions require a structured output — the agent must end every response with a `RECOMMENDATION:` line so we can parse the winner programmatically.

View your agents in [Studio](https://console.mistral.ai/build/agents).

In [16]:
agent = await client.beta.agents.create_async(
    name="Database Showdown Judge",
    description="Compares database candidates using their source code via DeepWiki.",
    model="mistral-large-latest",
    instructions=(
        "You are a database selection expert. "
        "Use the DeepWiki connectors to read each repository's source code and documentation. "
        "Evaluate: storage model, ACID guarantees, query capabilities, write throughput, and Python API simplicity. "
        "Be direct and data-driven. "
        "Always end your response with a line in exactly this format:\n"
        "RECOMMENDATION: <connector_name>\n"
        "where <connector_name> is one of: showdown_sqlite, showdown_duckdb, showdown_leveldb."
    ),
    tools=[
        {"type": "connector", "connector_id": connectors["showdown_sqlite"].id},
        {"type": "connector", "connector_id": connectors["showdown_duckdb"].id},
        {"type": "connector", "connector_id": connectors["showdown_leveldb"].id},
    ],
)
print(f"Agent ready: {agent.name}  (id={agent.id})")

Agent ready: Database Showdown Judge  (id=ag_019f8eb1b4cd76349125e8e58f662b2d)


## Step 4 — Run the comparison

Ask the agent to evaluate all three databases for a write-heavy local analytics workload. The agent will call DeepWiki tools on each Connector to read actual source code before answering — this may take a minute.

In [17]:
response = await client.beta.conversations.start_async(
    agent_id=agent.id,
    inputs=[
        {
            "role": "user",
            "content": (
                "Compare sqlite/sqlite, duckdb/duckdb, and google/leveldb for a write-heavy "
                "local analytics workload. Evaluate storage model, ACID guarantees, query "
                "capabilities, write throughput, and Python API simplicity. Recommend one."
            ),
        }
    ],
)

# Collect the agent's full reply
full_text = ""
for output in response.outputs:
    if output.type == "message.output":
        content = output.content
        if isinstance(content, str):
            full_text += content
        else:
            full_text += "".join(
                chunk.text if hasattr(chunk, "text") else str(chunk)
                for chunk in content
            )

print(full_text)

Here’s a direct, data-driven comparison of **SQLite**, **DuckDB**, and **LevelDB** for a **write-heavy local analytics workload**, based on their storage models, ACID guarantees, query capabilities, write throughput, and Python API simplicity:

---

### **1. Storage Model**
- **SQLite**: Row-oriented, disk-based (single-file). Optimized for transactional workloads but not columnar analytics.
- **DuckDB**: Columnar, in-memory-first with optional persistence. Optimized for analytical queries (vectorized execution).
- **LevelDB**: Key-value store (LSM-tree). Designed for high write throughput but not analytics (no native SQL).

**Winner for analytics**: DuckDB (columnar storage).

---

### **2. ACID Guarantees**
- **SQLite**: Full ACID (transactions, atomic writes, durability).
- **DuckDB**: Full ACID (transactions, atomic writes, durability).
- **LevelDB**: Atomic writes only (no multi-operation transactions).

**Winner for reliability**: SQLite/DuckDB (tie).

---

### **3. Query Capabil

In [ ]:
match = re.search(r"RECOMMENDATION:\s*([\w]+)", full_text)
if not match:
    raise ValueError("Agent did not return a RECOMMENDATION line — re-run the cell above.")

winner_name = match.group(1).strip()
loser_names = [name for name in connectors if name != winner_name]

print(f"Winner: {winner_name}")
print(f"Losers: {', '.join(loser_names)}")

## Step 5 — Promote the winner, retire the rest

Update the winning connector's description to mark it as selected, then delete the losing connectors. This completes the full lifecycle: create → list → use → update → delete.

In [19]:
# Mark the winner
winner_connector = connectors[winner_name]
updated = await client.beta.connectors.update_async(
    connector_id=winner_connector.id,
    description=f"[SELECTED] {winner_connector.description}",
)
print(f"Updated:  {updated.name}  —  {updated.description}")

# Delete the losers
for name in loser_names:
    result = await client.beta.connectors.delete_async(connector_id=connectors[name].id)
    print(f"Deleted:  {name}  —  {result.message}")

KeyError: 'showdown_duckdb**'

In [ ]:
# Confirm the winner is still there with its updated description
winner = await client.beta.connectors.get_async(connector_id_or_name=winner_name)
print("Winner confirmed:")
print(f"  Name:        {winner.name}")
print(f"  Description: {winner.description}")
print(f"  ID:          {winner.id}")

## Cleanup

Delete the agent when you're done. Uncomment the last two lines to also remove the winning Connector.

In [ ]:
await client.beta.agents.delete_async(agent_id=agent.id)
print(f"Agent deleted: {agent.id}")

# Uncomment to also remove the winning Connector:
# result = await client.beta.connectors.delete_async(connector_id=winner_connector.id)
# print(f"Connector deleted: {winner_name}")

## Summary

This notebook demonstrated the full Mistral Connector lifecycle — create, list, use, update, and delete — using the DeepWiki Connector to let the model read actual GitHub repository source code and produce a data-driven database recommendation.

**What you built:**
- Three named Connectors pointing at the DeepWiki MCP server
- An agent (Database Showdown Judge) with all three Connectors attached
- A conversation that produced a structured recommendation, updated the winner's Connector, and cleaned up the rest

**Mistral features used:**
- Connectors (beta)
- Agents API (beta)
- Conversations API (beta)

**Other services:**
- [DeepWiki](https://deepwiki.com) — MCP server for reading public GitHub repositories

View your Connectors in [Studio](https://console.mistral.ai/build/connectors).